In [1]:
import base64
from io import BytesIO
from PIL import Image, ImageEnhance

def preprocess_image(image_path, max_width=1024, do_enhance=True, return_base64=False):
    image = Image.open(image_path)
    
    # 1. Convert to grayscale
    gray_image = image.convert('L')
    
    # 2. Resize maintaining aspect ratio
    if gray_image.width > max_width:
        ratio = max_width / float(gray_image.width)
        new_height = int(gray_image.height * ratio)
        gray_image = gray_image.resize((max_width, new_height), Image.LANCZOS)

    # 3. Enhance contrast
    if do_enhance:
        enhancer = ImageEnhance.Contrast(gray_image)
        gray_image = enhancer.enhance(1.5)

    if return_base64:
        buffered = BytesIO()
        gray_image.save(buffered, format="JPEG", optimize=True, quality=95)
        img_str = base64.b64encode(buffered.getvalue()).decode('utf-8')
        return f"data:image/jpeg;base64,{img_str}"
    
    return gray_image

In [2]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import torch

# Load model locally or from HF
model_path = "Misraj/Baseer__Nakba"  # or your local path

processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForImageTextToText.from_pretrained(model_path)

# Load your local image
image_path = r"E:\Anaconda3\envs\Judge-Assistant\Case Sample\Case_Sample(4).jpeg"
image = Image.open(image_path).convert("RGB")

# Prepare message
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},  # 🔥 THIS changed
            {"type": "text", "text": "What is written in this document?"}
        ]
    },
]

# Process input
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

# Generate output
outputs = model.generate(**inputs, max_new_tokens=500)

# Decode result
result = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:])
print(result)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

دعوى الغاء قرار إزالة <|im_end|>


In [3]:
from transformers import pipeline

pipe = pipeline("image-text-to-text", model="Misraj/Baseer__Nakba")

image = Image.open(r"E:\Anaconda3\envs\Judge-Assistant\Case Sample\Case_Sample(4).jpeg")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Extract text from this document"}
        ]
    },
]

print(pipe(text=messages))

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

: 

In [1]:
!pip install transformers qwen_vl_utils accelerate>=0.26.0 PEFT -U
!pip install -U bitsandbytes

   ---------------------------------------- 0.0/55.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/55.4 MB ? eta -:--:--
   ---------------------------------------- 0.5/55.4 MB 1.7 MB/s eta 0:00:33
    --------------------------------------- 0.8/55.4 MB 1.7 MB/s eta 0:00:33
    --------------------------------------- 1.0/55.4 MB 1.6 MB/s eta 0:00:34
    --------------------------------------- 1.3/55.4 MB 1.6 MB/s eta 0:00:33
   - -------------------------------------- 1.8/55.4 MB 1.6 MB/s eta 0:00:33
   - -------------------------------------- 2.1/55.4 MB 1.6 MB/s eta 0:00:33
   - -------------------------------------- 2.4/55.4 MB 1.6 MB/s eta 0:00:33
   -- ------------------------------------- 2.9/55.4 MB 1.6 MB/s eta 0:00:33
   -- ------------------------------------- 3.1/55.4 MB 1.6 MB/s eta 0:00:33
   -- ------------------------------------- 3.4/55.4 MB 1.6 MB/s eta 0:00:32
   -- ------------------------------------- 3.7/55.4 MB 1.6 MB/s eta 0:00:32
   --- ------

In [4]:
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch
import os
from qwen_vl_utils import process_vision_info



model_name = "NAMAA-Space/Qari-OCR-v0.3-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
                model_name,
                torch_dtype="auto",
                device_map="auto"
            )
processor = AutoProcessor.from_pretrained(model_name)
max_tokens = 2000

prompt = """You are a strict Arabic OCR engine. Your only job is to transcribe exactly what is written in this document image.

Rules:
- Do NOT correct spelling mistakes
- Do NOT fix grammar
- Do NOT normalize or change any words
- Do NOT add or remove any punctuation
- Transcribe EXACTLY what you see, character by character
- If something looks like a typo, transcribe it as-is"""
image_path = r"E:\Anaconda3\envs\Judge-Assistant\Case Sample\Case_Sample(4).jpeg"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": prompt},
        ],
    }
]
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")
generated_ids = model.generate(**inputs, max_new_tokens=max_tokens)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]
print(output_text)


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


<p>أماني عبد <b>الحميد</b> <b>إبراهيم</b> المحامية <b>بالنقطة</b> والإدارية العليا</p><br><p>ت: 13/12/2021</p><br><h3>دعوى <i>الغاء</i> قرار ازالة السيد الاستاذ المستشار / رئيس محكمة <b>القضاء</b> الإداري بالإسماعيلية .</h3><br><p>تحية طيبة وبعد،،،،</p><br><p>مقدمه لسيادتكم / محمد <b>محمد</b> ابراهيم سعيد <i>–</i> المقيم بالعقار رقم <u>77</u> شارع السكة الحديد سابقا <u>87</u> شارع <b>شحته</b> حاليا <b>والاستاذ/</b> <b>طارق</b> رضا محمد علي المحامي بالإسماعيلية.</p><br><h2><u>مض</u> <u>دد</u></h2><br><p>1- السيد/ محافظة <i>الاسماعيلية</i></p><br><p>2- السيد/ رئيس مركز ومدينة <b>الاسماعيلية</b></p><br><p>3- السيد/ رئيس <i>الإدارة</i> الهندسية بحي ثان <b>الاسماعيلية</b></p><br><p>4- السيد/ رئيس <b>حي</b> ثان <b>الاسماعيلية</b></p><br><p>5- السيد/ مدير عام مديرية الاسكان بالاسماعيلية</p><br><p>6- السيد/ <i>عميد</i> مامور قسم ثان <b>الاسماعيلية</b></p><br><h4>ويعلنوا جميعا بهيئة قضايا الدولة بالاسماعيلية</h4><br><h3>7- السيد / محمد احمد <u>علي</u> – المقاول – صاحب شركة محمد علي للمقاولات وا

<p>أماني عبد <b>الحميد</b> <b>إبراهيم</b> المحامية <b>بالنقطة</b> والإدارية العليا</p><br><p>ت: 12/12/2013 <i>04:15</i></p><br><h3>دعوى <i>الغاء</i> قرار ازالة السيد <i>الاستاذ</i> المستشار / رئيس محكمة القضاء الإداري بالإسماعيلية <i>.</i></h3><br><p>تحية طيبة وبعد،،،،</p><br><p>مقدمه لسيادتكم / محمد <b>محمد</b> ابراهيم سعيد <i>–</i> المقيم بالعقار رقم <u>77</u> شارع السكة الحديد سابقا <u>87</u> شارع <b>شحته</b> حاليا والاستاذ/ طارق رضا محمد علي المحامي بالإسماعيلية.</p><br><h2><u>مض</u> <u>دد</u></h2><br><p>1- السيد/ محافظ <i>الاسماعيلية</i></p><br><p>2- السيد/ رئيس مركز ومدينة <b>الاسماعيلية</b></p><br><p>3- السيد/ رئيس <i>الإدارة</i> الهندسية بحي ثان <b>الاسماعيلية</b></p><br><p>4- السيد/ رئيس <b>حي</b> ثان <b>الاسماعيلية</b></p><br><p>5- السيد/ مدير عام مديرية الاسكان بالاسماعيلية</p><br><p>6- السيد/ <i>عميد</i> مامور قسم ثان <b>الاسماعيلية</b></p><br><h4>ويعلنوا جميعا بهيئة قضايا الدولة بالاسماعيلية</h4><br><h3>7- السيد / محمد احمد <u>علي</u> – المقاول – صاحب شركة محمد علي للمقاولات والانشاء والاستثمارات العقارية <u>–</u> <b>والمقيم</b> <u>خلف</u> رضا حلمي <b>للمشويات</b> بشارع شبين الكوم <i>–</i> ثالث <i>الاسماعيلية.</i></h3><br><h2><u>الموضوع</u></h2><br><p>بموجب عقد الاجار المؤرخ <i>اول</i> <i>اغسطس</i> 1909 اجر <i>المدعي</i> <b>الثقة</b> بالدور الارضي بالعقار رقم <u>77</u> شارع السكة الحديد سابقا <u>87</u> شارع <b>شحته</b> حاليا عرايشية مصر <i>–</i> <b>ثان</b> <i>الاسماعيلية</i> ، <b>يقيم</b> <b>الطالب</b> <b>بتلك</b> <b>الثقة</b> من تاريخ عقد الاجار حتى <b>وقتنا</b> هذا حيث قضى فيها عمره كله <b>شياه</b> ورجولته <b>وكبره،</b> يقيم <b>الطالب</b> اقامة هادئة مستقرة بدون اي مشاكل <b>ويسدد</b> دائما <b>القيمة</b> الاجارية الخاصة به.</p><br><h3>لما فوجئ <b>الطالب</b> بوجود قرار ازالة رقم <u>5</u> لسنة <u>2000</u> والصادر <b>بازالة</b> العقار بالكامل حتى سطح الارض ،.</h3><br><h3>ولما كان قرار لجنة <i>التنظيمات</i> قد جاء <b>ببطريقة</b> مخالفة <u>للقانون</u> وقد انتفته السرية <b>التامة</b> ولم يتم <i>اعلان</i> <b>الطالب</b> باي من اجراءات <i>اللجنة</i> من اي اجراءاتها ، ان كان <i>المتظلم</i> او <i>اللجنة</i> ذاتها ، <b>وعدن</b> <i>تتبعه</i> للام فوجئ <b>بأنه</b> قد سبق قرار <i>الازالة</i> قرار ترميم <i>رقم</i> <u>4</u> لسنة 2019 ولم <i>يعلن</i> به ايضا <b>.</b></h3>


In [1]:
import torch
print(torch.cuda.is_available())    # should print True
print(torch.cuda.get_device_name(0))  # should print: NVIDIA GeForce RTX 4060

True
NVIDIA GeForce RTX 4060


<p>أماني عبد <b>الحميد</b> <b>إبراهيم</b> المحامية <b>بالنقطة</b> والإدارية العليا</p><br><p>ت: 13/12/2021</p><br><h3>دعوى <i>الغاء</i> قرار ازالة السيد الاستاذ المستشار / رئيس محكمة <b>القضاء</b> الإداري بالإسماعيلية .</h3><br><p>تحية طيبة وبعد،،،،</p><br><p>مقدمه لسيادتكم / محمد <b>محمد</b> ابراهيم سعيد <i>–</i> المقيم بالعقار رقم <u>77</u> شارع السكة الحديد سابقا <u>87</u> شارع <b>شحته</b> حاليا <b>والاستاذ/</b> <b>طارق</b> رضا محمد علي المحامي بالإسماعيلية.</p><br><h2><u>مض</u> <u>دد</u></h2><br><p>1- السيد/ محافظة <i>الاسماعيلية</i></p><br><p>2- السيد/ رئيس مركز ومدينة <b>الاسماعيلية</b></p><br><p>3- السيد/ رئيس <i>الإدارة</i> الهندسية بحي ثان <b>الاسماعيلية</b></p><br><p>4- السيد/ رئيس <b>حي</b> ثان <b>الاسماعيلية</b></p><br><p>5- السيد/ مدير عام مديرية الاسكان بالاسماعيلية</p><br><p>6- السيد/ <i>عميد</i> مامور قسم ثان <b>الاسماعيلية</b></p><br><h4>ويعلنوا جميعا بهيئة قضايا الدولة بالاسماعيلية</h4><br><h3>7- السيد / محمد احمد <u>علي</u> – المقاول – صاحب شركة محمد علي للمقاولات والانشاء والاستثمارات العقارية <u>–</u> <b>والمقيم</b> <u>خلف</u> رضا حلمي <b>للمشويات</b> بشارع شبين الكوم <i>–</i> ثالث <i>الاسماعيلية.</i></h3><br><h2><u>الموضوع</u></h2><br><p>بموجب عقد الاجار المؤرخ <i>اول</i> <i>اغسطس</i> 1909 اجر <b>المدعي</b> <b>الثقة</b> بالدور الارضي بالعقار رقم <u>77</u> شارع السكة الحديد سابقا <u>87</u> شارع <b>شحته</b> حاليا عرايشية مصر <i>–</i> <b>ثان</b> <i>الاسماعيلية</i> ، <b>يقيم</b> <b>الطالب</b> <i>بتلك</i> <b>الثقة</b> <u>من</u> تاريخ عقد الاجار حتى <b>وقتنا</b> هذا حيث قضى فيها عمره كله <b>شياه</b> ورجولته <b>وكبره،</b> يقيم <b>الطالب</b> اقامة هادئة مستقرة بدون اي مشاكل <b>ويسدد</b> دائما <b>القيمة</b> الاجارية الخاصة به.</p><br><h3>لما فوجئ <b>الطالب</b> بوجود قرار ازالة رقم <u>5</u> لسنة <u>2020</u> والصادر <b>بازالة</b> العقار بالكامل حتى سطح الارض ،.</h3><br><p>ولما كان قرار لجنة <i>التنظيمات</i> قد جاء <b>ببطريقة</b> مخالفة <u>للقانون</u> وقد انتفته السرية <b>التامة</b> ولم يتم <i>اعلان</i> <b>الطالب</b> باي من اجراءات <i>اللجنة</i> من اي اجراءاتها ، ان كان <i>المتظلم</i> او <i>اللجنة</i> ذاتها ، <b>وعدن</b> <i>تتبعه</i> للام فوجئ <b>بأنه</b> قد سبق قرار <i>الازالة</i> قرار ترميم <i>رقم</i> <u>4</u> لسنة 2019 ولم <i>يعلن</i> به ايضا .</p>